# GreenMind AI: Notebook 6, lite engine (ONNX) vs full engine (PyTorch)

**Settings:** Accelerator = **None (CPU)**, Internet = **On**.
**Inputs (Add Input → Notebook):** `GreenMind AI: Notebook 5` (model bundle) and `GreenMind AI: Notebook 2` (`inat.tar`).
Then **Save Version → Save & Run All (Commit)**, about 1.5–2 h.

1. Exports the species and disease classifiers and both YOLO detectors to ONNX (`/kaggle/working/onnx`, download it for the app).
2. Accuracy of both engines on the same images: species on Pool B (18,923 photos), disease on PlantDoc (102 photos), plus per-image agreement.
3. Resident memory, model-loading time and CPU latency (2 threads) of each engine, and installed package size.

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("greenmind-ai"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "lite-onnx", "https://github.com/mubashir20023/greenmind-ai"], check=True)
os.chdir("/kaggle/working/greenmind-ai")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.19", "ultralytics==8.3.187",
                "onnx", "onnxruntime", "onnxslim", "psutil", "scikit-learn"], check=False)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
if not os.path.exists("/kaggle/working/PlantDoc-Dataset"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pratikkayal/PlantDoc-Dataset", "/kaggle/working/PlantDoc-Dataset"], check=True)

In [ ]:
import glob, os, urllib.request
A = os.path.dirname(sorted(glob.glob("/kaggle/input/**/app_models/plant_id_effb0_best.pth", recursive=True))[0])
TAR = sorted(glob.glob("/kaggle/input/**/inat.tar", recursive=True))[0]
print(A, TAR)
organ = "runs/detect/train/weights/best.pt"
if os.path.getsize(organ) < 10_000:
    urllib.request.urlretrieve("https://media.githubusercontent.com/media/mubashir20023/greenmind-ai/main/" + organ, organ)
os.makedirs("/kaggle/temp", exist_ok=True)
if not os.path.exists("/kaggle/temp/inat/pool_b"):
    !tar -xf "{TAR}" -C /kaggle/temp inat/pool_b
PD = "/kaggle/working/PlantDoc-Dataset/test"
LM = "tools/plantdoc_label_map.json"
ONNX = "/kaggle/working/onnx"

In [ ]:
# 1) Export to ONNX (with a numerical check against PyTorch)
!python tools/export_onnx.py --species_ckpt {A}/plant_id_effb0_best.pth --species_classes {A}/idx_to_species_effb0.json \
    --disease_ckpt {A}/leaf_health_effb0_field.pth --disease_classes {A}/classes.json \
    --organ_yolo {organ} --leaf_yolo {A}/yolo_leaf.pt --out {ONNX}

In [ ]:
# 2) Same images through both engines
!python tools/evaluate.py --ckpt {A}/plant_id_effb0_best.pth --arch efficientnet_b0 --classes {A}/idx_to_species_effb0.json \
    --imagefolder /kaggle/temp/inat/pool_b --workers 4 --device cpu --no_latency --out /kaggle/working/cmp/torch_species
!ONNX_DIR={ONNX} python tools/compare_engines.py accuracy --task species --imagefolder /kaggle/temp/inat/pool_b \
    --out /kaggle/working/cmp/onnx_species
!python tools/evaluate.py --ckpt {A}/leaf_health_effb0_field.pth --arch efficientnet_b0 --norm half --resize_mode square \
    --classes {A}/classes.json --imagefolder {PD} --label_map {LM} --device cpu --no_latency --out /kaggle/working/cmp/torch_disease
!ONNX_DIR={ONNX} python tools/compare_engines.py accuracy --task disease --imagefolder {PD} --label_map {LM} \
    --out /kaggle/working/cmp/onnx_disease

In [ ]:
# 3) Memory, loading time, latency (2 CPU threads), package size
img = sorted(glob.glob(PD + "/Tomato leaf late blight/*"))[0]
!python tools/compare_engines.py resources --engine torch --image "{img}" --threads 2 \
    --species_ckpt {A}/plant_id_effb0_best.pth --species_classes {A}/idx_to_species_effb0.json \
    --disease_ckpt {A}/leaf_health_effb0_field.pth --disease_classes {A}/classes.json \
    --organ_yolo {organ} --leaf_yolo {A}/yolo_leaf.pt --out /kaggle/working/cmp/resources_torch.json
!ONNX_DIR={ONNX} python tools/compare_engines.py resources --engine onnx --image "{img}" --threads 2 \
    --out /kaggle/working/cmp/resources_onnx.json
!python tools/compare_engines.py packages --out /kaggle/working/cmp/packages.json

In [ ]:
# Summary
import csv, json
def preds(path):
    with open(path, newline="", encoding="utf-8") as f:
        return {r["path"]: r["pred"] for r in csv.DictReader(f)}
summary = {"export": json.load(open(f"{ONNX}/export_report.json"))}
for task in ["species", "disease"]:
    t, o = f"/kaggle/working/cmp/torch_{task}", f"/kaggle/working/cmp/onnx_{task}"
    pt, po = preds(f"{t}/predictions.csv"), preds(f"{o}/predictions.csv")
    common = set(pt) & set(po)
    mt, mo = json.load(open(f"{t}/metrics.json")), json.load(open(f"{o}/metrics.json"))
    summary[task] = {"n": len(common), "torch_top1": mt["top1"], "onnx_top1": mo["top1"],
                     "torch_top5": mt.get("top5"), "onnx_top5": mo["top5"],
                     "top1_prediction_agreement": sum(pt[p] == po[p] for p in common) / max(1, len(common))}
for k in ["resources_torch", "resources_onnx", "packages"]:
    summary[k] = json.load(open(f"/kaggle/working/cmp/{k}.json"))
json.dump(summary, open("/kaggle/working/cmp/engine_comparison.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
!du -sh {ONNX}; ls -la {ONNX}